# 🔬 Auditoría de la Cohorte de 3,848 Aislamientos
Este notebook reconstruye la trazabilidad exacta de los 3,848 aislamientos que componen el dataset limpio, confirmando su origen, verificando el split Train/Validation (sin solapamiento), evaluando la disponibilidad de metadatos clínicos y analizando los 146 aislamientos que no pasaron a la etapa de *ST-blocked*.

Al final, genera un archivo Excel maestro `Cohort_3848_Audit.xlsx` con todas las hojas de auditoría.

## 1. Configuración de Entorno y Rutas

In [ ]:
import os, re
import pandas as pd
import numpy as np
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive')

# Rutas de los archivos necesarios en Google Drive
# ⚠️ MODIFICA ESTAS RUTAS SEGÚN TU ESTRUCTURA EN DRIVE ⚠️
BASE_DIR = '/content/drive/MyDrive/rgi_Klebsiella_pneumoniae_ertapenem/dataset_limpio'
METADATA_DIR = '/content/drive/MyDrive/rgi_Klebsiella_pneumoniae_ertapenem/baterias' # Ajustar donde estén los excels

PATHS = {
    'train': os.path.join(BASE_DIR, 'train_por_cepa.csv'),
    'val': os.path.join(BASE_DIR, 'val_por_cepa.csv'),
    'oof': os.path.join(BASE_DIR, 'oof_predictions_pooled.csv'),  # Asumiendo que está guardado aquí
    'cepasv2': os.path.join(METADATA_DIR, 'cepasv2.xlsx'),
    'auditoria': os.path.join(METADATA_DIR, 'Auditoria_Final_Armonizada.xlsx')
}

# Verificación de archivos
for name, p in PATHS.items():
    if os.path.exists(p):
        print(f"✅ Encontrado: {name} -> {p}")
    else:
        print(f"❌ Falta: {name} -> {p}")


## 2. Verificación del Split Train / Validation (3,848 cepas)

In [ ]:
# Cargar datos
try:
    df_train = pd.read_csv(PATHS['train'])
    df_val = pd.read_csv(PATHS['val'])
except FileNotFoundError:
    raise Exception("Por favor, sube o ajusta las rutas de train_por_cepa.csv y val_por_cepa.csv")

# Extraer aislamientos únicos y conteos
train_seqs = len(df_train)
val_seqs = len(df_val)

df_train_cepas = df_train[['genome_id', 'label']].drop_duplicates()
df_val_cepas = df_val[['genome_id', 'label']].drop_duplicates()

train_gids = set(df_train_cepas['genome_id'])
val_gids = set(df_val_cepas['genome_id'])

# Solapamiento
overlap = train_gids.intersection(val_gids)
total_gids = train_gids.union(val_gids)

print("=== VERIFICACIÓN DE LA COHORTE BASE ===")
print(f"Train secuencias: {train_seqs:,} | Train aislamientos únicos: {len(train_gids):,}")
print(f"Val secuencias  : {val_seqs:,} | Val aislamientos únicos  : {len(val_gids):,}")
print(f"Total secuencias: {train_seqs + val_seqs:,}")
print(f"Total aislamientos únicos: {len(total_gids):,}")
print(f"Solapamiento Train ∩ Val : {len(overlap)} (DEBE SER 0)")

# Crear base para la tabla maestra
df_train_cepas['split'] = 'Train'
df_val_cepas['split'] = 'Validation'
df_3848 = pd.concat([df_train_cepas, df_val_cepas], ignore_index=True)

# Calcular n_sequences por aislamiento
seq_counts = pd.concat([df_train['genome_id'], df_val['genome_id']]).value_counts().reset_index()
seq_counts.columns = ['genome_id', 'n_sequences']
df_3848 = df_3848.merge(seq_counts, on='genome_id')


## 3. Reconstrucción de Procedencia y Trazabilidad (Metadata)

In [ ]:
# Cargar metadatos si existen
try:
    df_cepasv2 = pd.read_excel(PATHS['cepasv2'])
    df_audit = pd.read_excel(PATHS['auditoria'])
    metadata_available = True
except FileNotFoundError:
    print("⚠️ Faltan archivos de metadatos. La trazabilidad será parcial.")
    df_cepasv2 = pd.DataFrame()
    df_audit = pd.DataFrame()
    metadata_available = False

# Mapear cepasv2 (BV-BRC/PATRIC)
cepasv2_map = {}
if not df_cepasv2.empty:
    for _, row in df_cepasv2.iterrows():
        gid = str(row.get('genome_id', '')).strip()
        if gid and gid != 'nan':
            # Limpiar para match (573.XXXXX -> 573_XXXXX)
            clean_id = gid.replace('.', '_')
            cepasv2_map[clean_id] = row

# Mapear Auditoria (Zenodo)
audit_map = {}
if not df_audit.empty:
    audit_gid_col = 'genome_id' if 'genome_id' in df_audit.columns else df_audit.columns[0]
    for _, row in df_audit.iterrows():
        gid = str(row.get(audit_gid_col, '')).strip()
        if gid and gid != 'nan':
            audit_map[gid] = row

master_data = []

for _, row in df_3848.iterrows():
    gid = row['genome_id']
    country = np.nan
    collection_date = np.nan
    testing_standard = np.nan
    accession = np.nan
    source = 'Unknown / Other'
    
    # 1. Intentar match con PATRIC (BV-BRC) - patrón 573_XXXXX
    m_patric = re.search(r'(573_\d+)$', gid)
    # 2. Intentar extracción de GCA
    m_gca = re.search(r'(GC[AF]_\d+(?:\.\d+)?)', gid)
    gca = m_gca.group(1) if m_gca else None
    
    # Evaluar procedencia
    if m_patric and m_patric.group(1) in cepasv2_map:
        source = 'BV-BRC / PATRIC (cepasv2)'
        r = cepasv2_map[m_patric.group(1)]
        country = r.get('country', np.nan)
        collection_date = r.get('collection_date', np.nan)
        testing_standard = r.get('testing_standard', np.nan)
        accession = r.get('assembly_accession', gca)
        
    elif gca and gca in audit_map:
        source = 'Zenodo (Auditoria Original)'
        r = audit_map[gca]
        accession = gca
        
    elif 'euscape' in gid.lower():
        source = 'EuSCAPE (Metadata)'
        accession = gca if gca else np.nan
        
    # Construir fila
    pheno_text = 'Resistant' if row['label'] == 1 else 'Susceptible'
    
    master_data.append({
        'genome_id': gid,
        'accession': accession,
        'source': source,
        'source_file': f"{gid}.txt",
        'country': country,
        'collection_date': collection_date,
        'phenotype': pheno_text,
        'MIC': np.nan,  # Valor no disponible directamente en las DB exportadas
        'testing_method': np.nan,
        'testing_standard': testing_standard,
        'ST': np.nan,  # Se asigna post-Kleborate
        'split': row['split'],
        'n_sequences': row['n_sequences'],
        'inclusion_status': 'In 3,848 cohort'
    })

df_master = pd.DataFrame(master_data)

print(f"Resumen de Fuentes de los {len(df_master):,} aislamientos:")
print(df_master['source'].value_counts())


## 4. Comparación con Análisis ST-Blocked (Los 146 faltantes)

In [ ]:
# Cargar predicciones OOF (contienen los IDs de los 3,702 del análisis ST-blocked)
try:
    df_oof = pd.read_csv(PATHS['oof'])
    oof_gids = set(df_oof['genome_id'])
    oof_available = True
    print(f"OOF (ST-blocked) genomes: {len(oof_gids):,}")
except FileNotFoundError:
    print("⚠️ No se encontró oof_predictions_pooled.csv. Saltando comparación de 146.")
    oof_available = False
    oof_gids = set()

if oof_available:
    missing_146_ids = total_gids - oof_gids
    print(f"Diferencia (3848 - 3702): {len(missing_146_ids)} aislamientos")
    
    # Categorizar los 146
    missing_data = []
    
    # Mapeo de GCAs en OOF (para detectar duplicados técnicos)
    oof_gcas = set()
    for ogid in oof_gids:
        m = re.search(r'(GC[AF]_\d+(?:\.\d+)?)', ogid)
        if m: oof_gcas.add(m.group(1))
            
    for gid in missing_146_ids:
        # Extraer GCA corregido (manejo del bug de los 11 dígitos EuSCAPE concatenados)
        m_gca = re.search(r'(GC[AF]_\d+)', gid)
        gca = None
        category = 'Unknown'
        reason = ''
        
        if m_gca:
            raw_gca = m_gca.group(1)
            # Fix EuSCAPE concatenado (ej. GCA_9005065551 -> GCA_900506555.1)
            if len(raw_gca) == 14 and raw_gca.endswith('1'): # GCA_ + 10 digits
                gca = f"GCA_{raw_gca[4:13]}.1"
            else:
                gca = raw_gca + '.1' # simplificación
                
            # Verificar si este GCA ya está en el OOF bajo otro nombre limpio
            if any(gca.replace('.1','') in ogca for ogca in oof_gcas):
                category = 'Technical Duplicate (Do not recover)'
                reason = 'Same GCA is already present in 3,702 under a clean accession ID'
            else:
                category = 'EuSCAPE (Potentially Recoverable)'
                reason = 'Missing ST assignment due to filename-accession mismatch in Kleborate run'
        else:
            category = 'Unresolvable (No GCA)'
            reason = 'Cannot extract NCBI accession to run Kleborate (e.g. GIMC/ARGID format)'
            
        pheno = 'Resistant' if df_master[df_master['genome_id']==gid]['phenotype'].iloc[0] == 'Resistant' else 'Susceptible'
        
        missing_data.append({
            'genome_id': gid,
            'extracted_gca': gca if gca else np.nan,
            'phenotype': pheno,
            'category': category,
            'exclusion_reason': reason
        })
        
        # Actualizar master list con la razón de exclusión
        df_master.loc[df_master['genome_id'] == gid, 'inclusion_status'] = 'Excluded from ST-blocked (3702)'
        df_master.loc[df_master['genome_id'] == gid, 'exclusion_reason'] = reason

    df_missing_146 = pd.DataFrame(missing_data)
    print("
Categorización de los 146 faltantes:")
    print(df_missing_146['category'].value_counts())
else:
    df_missing_146 = pd.DataFrame()


## 5. Auditoría de Completitud y Generación del Excel Final

In [ ]:
# Calcular Metadata Completeness
total = len(df_master)
completeness = []
for col in df_master.columns:
    na_count = df_master[col].isna().sum()
    completeness.append({
        'Variable': col,
        'N_Available': total - na_count,
        'N_Missing': na_count,
        'Percent_Missing': f"{(na_count/total)*100:.1f}%"
    })
df_completeness = pd.DataFrame(completeness)

print("=== METADATA COMPLETENESS ===")
print(df_completeness.to_string(index=False))

# Texto README
readme_text = pd.DataFrame({'README': [
    "AUDITORÍA DE LA COHORTE 3,848 AISLAMIENTOS",
    "==========================================",
    "Esta hoja documenta la trazabilidad exacta de los 3,848 aislamientos usados para entrenar los modelos.",
    "",
    "HOJAS:",
    "1. Cohort_3848: Master table con los 3,848 genomas, su split y procedencia.",
    "2. Source_summary: Conteo de fuentes (PATRIC, Zenodo, EuSCAPE).",
    "3. Train_Validation: Conteo confirmando el split.",
    "4. Missing_146: Análisis de los 146 excluidos del ST-blocked.",
    "5. Duplicates: Listado de los duplicados técnicos encontrados en los 146.",
    "6. Metadata_completeness: Porcentaje de completitud por variable clínica."
]})

# Summary tablas
df_source_summary = df_master['source'].value_counts().reset_index()
df_source_summary.columns = ['Source', 'Count']

df_split_summary = df_master.groupby(['split', 'phenotype']).size().unstack(fill_value=0).reset_index()
df_split_summary['Total'] = df_split_summary['Resistant'] + df_split_summary['Susceptible']

# Exportar a Excel
output_excel = os.path.join(BASE_DIR, 'Cohort_3848_Audit.xlsx') if os.path.exists(BASE_DIR) else 'Cohort_3848_Audit.xlsx'

with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
    readme_text.to_excel(writer, sheet_name='README', index=False, header=False)
    df_master.to_excel(writer, sheet_name='Cohort_3848', index=False)
    df_source_summary.to_excel(writer, sheet_name='Source_summary', index=False)
    df_split_summary.to_excel(writer, sheet_name='Train_Validation', index=False)
    
    if not df_missing_146.empty:
        df_missing_146.to_excel(writer, sheet_name='Missing_146', index=False)
        # Hoja separada para duplicados
        dupes = df_missing_146[df_missing_146['category'] == 'Technical Duplicate (Do not recover)']
        dupes.to_excel(writer, sheet_name='Duplicates', index=False)
        
    df_completeness.to_excel(writer, sheet_name='Metadata_completeness', index=False)

print(f"\n✅ Auditoría completa. Archivo Excel generado: {output_excel}")
